# Etape 3 - Robustesse a l'intensite de degradation

On evalue la **robustesse au test** des 5 modeles deja entraines (3 baselines +
Strat1 + Strat2), sur **Animals-10** et **Imagewoof**.

- Les modeles sont charges depuis leurs checkpoints `.pth` (entraines sur le BF
  canonique : downsample 64px + bruit sigma=0.15 + JPEG q60). **Aucun re-entrainement.**
- On les evalue sur des jeux de **test degrades a intensites croissantes**, via le
  module partage `src/degradation.py` (parametres `downscale`, `sigma`, `jpeg_quality`).

Deux balayages 1-D :
1. **Bruit** : sigma variable, sous-echantillonnage fixe a 64px.
2. **Sous-echantillonnage** : resolution intermediaire variable, sigma fixe a 0.15.

JPEG fixe a q60. Sorties : 1 JSON + 2 figures (une par dataset, 2 panneaux) dans `results/comparison/`.

> Prerequis : les checkpoints doivent provenir du **nouveau pipeline** (post correction
> du decalage de domaine). Lancer d'abord les baselines (nb03/08) et Strat1/Strat2
> (nb04/05/09/10) AVANT ce notebook.

## 1) Imports, Drive et module de degradation

In [ ]:
import os
import sys
import json
import time
import zipfile
import importlib

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models
import matplotlib.pyplot as plt

# Drive (Colab) ou execution locale
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Module de degradation partage (source unique)
_SRC_CANDIDATES = ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']
for _p in _SRC_CANDIDATES:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import degradation
importlib.reload(degradation)
from degradation import DegradedDataset, clean_tensor_transform, hf_transform
import cost
importlib.reload(cost)
from cost import unit_cost
import env_config
importlib.reload(env_config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Appareil:', device)
print('Module degradation:', os.path.dirname(degradation.__file__))

## 2) Configuration : datasets, modeles, niveaux de degradation

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2
NUM_CLASSES = 10
JPEG_QUALITY = 60

SIGMA_LEVELS = [0.0, 0.05, 0.10, 0.15, 0.25, 0.35]
SIGMA_FIXED_DOWNSCALE = 64
DOWNSCALE_LEVELS = [224, 128, 96, 64, 48, 32]
DOWNSCALE_FIXED_SIGMA = 0.15

MODELS = [
    ('BL 1(HF)',          'model_baseline_HF.pth',                       '#4C72B0'),
    ('BL 2(BF)',          'model_baseline_BF.pth',                       '#55A868'),
    ('BL 3(MIXTE)',       'model_baseline_MIXTE.pth',                    '#C44E52'),
    ('Strat 1(BF->HF)',   'model_strategy1_transfer_learning.pth',      '#8172B2'),
    ('Strat 2(CoTrain)',  'model_strategy2_cotraining_reweighting.pth', '#CCB974'),
]

# Colab : extraire les 2 zips dans des dossiers separes. Serveur/local : dossiers data/ deja prets.
if env_config.in_colab():
    DRIVE = '/content/drive/MyDrive/UTBM_PF22'
    DATASETS = [
        {'name': 'Animals-10', 'zip': f'{DRIVE}/datasets/Animals-10/dataset_multifidelity.zip',
         'results': env_config.results_dir('Animals-10'), 'local': '/content/mf_animals'},
        {'name': 'Imagewoof',  'zip': f'{DRIVE}/datasets/Imagewoof/dataset_multifidelity.zip',
         'results': env_config.results_dir('Imagewoof'), 'local': '/content/mf_imagewoof'},
    ]
else:
    DATASETS = [
        {'name': 'Animals-10', 'zip': None, 'results': env_config.results_dir('Animals-10'),
         'local': env_config.data_dir('Animals-10')},
        {'name': 'Imagewoof',  'zip': None, 'results': env_config.results_dir('Imagewoof'),
         'local': env_config.data_dir('Imagewoof')},
    ]
OUT_DIR = env_config.comparison_dir()
os.makedirs(OUT_DIR, exist_ok=True)
print('Ratios/niveaux configures. Sortie :', OUT_DIR)

## 3) Utilitaires : extraction test, chargement modele, evaluation

In [ ]:
def ensure_test_dir(ds):
    """Retourne le dossier test/ du dataset, en extrayant le zip si necessaire."""
    local = ds['local']
    candidates = [
        os.path.join(local, 'processed_multifidelity', 'test'),
        os.path.join(local, 'test'),
    ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    if ds.get('zip') and os.path.exists(ds['zip']):
        os.makedirs(local, exist_ok=True)
        print(f"  Extraction {ds['name']} ...")
        with zipfile.ZipFile(ds['zip'], 'r') as zf:
            zf.extractall(local)
        for c in candidates:
            if os.path.isdir(c):
                return c
    raise FileNotFoundError(f"Dossier test introuvable pour {ds['name']} (cherche: {candidates})")


def load_model(path):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    return model.to(device).eval()


@torch.no_grad()
def eval_accuracy(model, test_dir, downscale, sigma, jpeg_quality):
    base = datasets.ImageFolder(test_dir, transform=clean_tensor_transform())
    ds = DegradedDataset(base, downscale=downscale, sigma=sigma,
                         jpeg_quality=jpeg_quality, seeded=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total


@torch.no_grad()
def eval_clean(model, test_dir):
    """Precision sur le test HF totalement propre (reference)."""
    base = datasets.ImageFolder(test_dir, transform=hf_transform())
    loader = DataLoader(base, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

## 4) Execution des balayages (5 modeles x 2 datasets)

In [ ]:
results = {}
t0 = time.time()

for ds in DATASETS:
    name = ds['name']
    print(f"\n===== {name} =====")
    try:
        test_dir = ensure_test_dir(ds)
    except FileNotFoundError as e:
        print('  SKIP', name, ':', e)
        continue

    res_dir = ds['results']
    ds_res = {
        'sigma_levels': SIGMA_LEVELS,
        'sigma_fixed_downscale': SIGMA_FIXED_DOWNSCALE,
        'downscale_levels': DOWNSCALE_LEVELS,
        'downscale_fixed_sigma': DOWNSCALE_FIXED_SIGMA,
        'jpeg_quality': JPEG_QUALITY,
        'downscale_unit_costs': [round(unit_cost(d), 3) for d in DOWNSCALE_LEVELS],
        'models': {},
    }
    print('  Couts unitaires (CA) par resolution:',
          {d: round(unit_cost(d), 2) for d in DOWNSCALE_LEVELS})

    for disp, fname, color in MODELS:
        path = os.path.join(res_dir, fname)
        if not os.path.exists(path):
            print(f'  [absent] {disp}: {path}')
            continue
        model = load_model(path)
        clean_acc = eval_clean(model, test_dir)
        sigma_acc = [eval_accuracy(model, test_dir, SIGMA_FIXED_DOWNSCALE, s, JPEG_QUALITY)
                     for s in SIGMA_LEVELS]
        down_acc = [eval_accuracy(model, test_dir, d, DOWNSCALE_FIXED_SIGMA, JPEG_QUALITY)
                    for d in DOWNSCALE_LEVELS]
        ds_res['models'][disp] = {
            'color': color,
            'clean_HF_acc': clean_acc,
            'sigma_acc': sigma_acc,
            'downscale_acc': down_acc,
        }
        print(f"  {disp:18} cleanHF={clean_acc:5.2f} | "
              f"sigma={['%.1f' % v for v in sigma_acc]} | "
              f"down={['%.1f' % v for v in down_acc]}")

    results[name] = ds_res

json_path = os.path.join(OUT_DIR, 'robustness_degradation.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nTermine en {(time.time() - t0) / 60:.1f} min. JSON sauve : {json_path}")

## 5) Figures : precision vs intensite (une figure par dataset)

In [ ]:
for ds in DATASETS:
    name = ds['name']
    if name not in results or not results[name]['models']:
        print('Pas de resultats pour', name)
        continue
    r = results[name]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    fig.suptitle(f"Robustesse a l'intensite de degradation - {name}", fontsize=14)

    # Panneau 1 : bruit gaussien (downsample fixe)
    ax = axes[0]
    for disp, m in r['models'].items():
        ax.plot(r['sigma_levels'], m['sigma_acc'], marker='o', color=m['color'], label=disp)
    ax.set_title(f"Bruit gaussien (downsample fixe = {r['sigma_fixed_downscale']}px, JPEG q{r['jpeg_quality']})")
    ax.set_xlabel('Sigma du bruit gaussien (plus a droite = plus degrade)')
    ax.set_ylabel('Precision Test (%)')
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    # Panneau 2 : sous-echantillonnage (sigma fixe). x ascendant -> gauche = plus degrade.
    ax = axes[1]
    for disp, m in r['models'].items():
        ax.plot(r['downscale_levels'], m['downscale_acc'], marker='s', color=m['color'], label=disp)
    ax.set_title(f"Sous-echantillonnage (sigma fixe = {r['downscale_fixed_sigma']}, JPEG q{r['jpeg_quality']})")
    ax.set_xlabel('Resolution intermediaire (px) -- gauche = plus degrade')
    ax.set_ylabel('Precision Test (%)')
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    suffix = 'animals' if name == 'Animals-10' else 'imagewoof'
    out = os.path.join(OUT_DIR, f'robustness_degradation_{suffix}.png')
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
    print('Figure sauvee:', out)

## 6) Notes

- **Lecture** : une courbe par modele. Une courbe qui descend lentement = modele
  robuste a la degradation ; une chute brutale = modele fragile au decalage de domaine.
- Le point `sigma=0` / `downsample=224` correspond a la degradation la plus faible
  de chaque axe (mais pas totalement propre : JPEG q60 reste applique). La precision
  HF totalement propre est stockee dans le JSON (`clean_HF_acc`).
- Pour ajouter ces figures au rapport : copier
  `results/comparison/robustness_degradation_animals.png` et `..._imagewoof.png`
  dans `latek_report/figures/`.